# 01 - Data Understanding & Cleaning
Goal: profile the raw Telco data, fix its quirks deliberately, and document *why*.

In [1]:
import sys; sys.path.append('..')
import pandas as pd
from src.clean import load_raw, clean
pd.set_option('display.width', 140)
raw = load_raw()
raw.shape

(7043, 21)

## 1. Profile

In [2]:
raw.dtypes.value_counts(), raw.isna().sum().sum(), raw.customerID.is_unique

(object     18
 int64       2
 float64     1
 Name: count, dtype: int64,
 np.int64(0),
 True)

In [3]:
raw.select_dtypes('object').nunique().sort_values()

gender                 2
Partner                2
Dependents             2
PhoneService           2
PaperlessBilling       2
Churn                  2
OnlineSecurity         3
InternetService        3
StreamingTV            3
OnlineBackup           3
TechSupport            3
DeviceProtection       3
Contract               3
StreamingMovies        3
MultipleLines          3
PaymentMethod          4
TotalCharges        6531
customerID          7043
dtype: int64

No nulls and no duplicate IDs, and all categoricals have low cardinality (max 4 levels, apart from the ID). The only red flag is `TotalCharges` being stored as a string.

## 2. Target balance

In [4]:
raw.Churn.value_counts(normalize=True).round(3)

Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64

**~26.5% churn, a 2.8:1 class imbalance.** A model that predicts 'no churn' for everyone scores ~73% accuracy and is useless, so Phase 5 evaluates with precision/recall/ROC-AUC/PR-AUC and handles imbalance explicitly.

## 3. The `TotalCharges` quirk

In [5]:
tc = pd.to_numeric(raw.TotalCharges, errors='coerce')
raw[tc.isna()][['customerID','tenure','MonthlyCharges','Contract','Churn']]

,customerID,tenure,MonthlyCharges,Contract,Churn
488,4472-LVYGI,0,52.55,Two year,No
753,3115-CZMZD,0,20.25,Two year,No
936,5709-LVOEQ,0,80.85,Two year,No
1082,4367-NUYAO,0,25.75,Two year,No
1340,1371-DWPAZ,0,56.05,Two year,No
3331,7644-OMVMY,0,19.85,Two year,No
3826,3213-VVOLG,0,25.35,Two year,No
4380,2520-SGTTA,0,20.00,Two year,No
5218,2923-ARZLG,0,19.70,One year,No
6670,4075-WKNIU,0,73.35,Two year,No


**Why I handled it this way.** All 11 blank rows have `tenure == 0`: they signed up but have not had a first bill. The true total charged is therefore **0**, not 'missing'.

- *Dropping* would silently remove new customers (and all 11 are non-churners).
- *Mean/median imputation* would invent about $2,000 of billing history for people who have paid nothing.
- *Setting to 0* is the factually correct value, and the cleaning code asserts that every blank row has tenure 0, so the pipeline fails loudly if that assumption ever breaks.

Side effect to remember: `tenure == 0` customers cannot enter a survival model (duration 0), so Phase 6 excludes them.

## 4. Redundant category levels

In [6]:
raw[['MultipleLines','OnlineSecurity','StreamingTV']].apply(lambda s: s.value_counts()).T

,No,No internet service,No phone service,Yes
MultipleLines,3390.0,NaN,682.0,2971.0
OnlineSecurity,3498.0,1526.0,NaN,2019.0
StreamingTV,2810.0,1526.0,NaN,2707.0


'No internet service' and 'No phone service' are already encoded by `InternetService` / `PhoneService`. Keeping them as a third level for each add-on inflates one-hot dimensionality and creates collinearity, so I collapse them to 'No'. Churn target becomes 0/1.

## 5. Clean and save

In [7]:
df = clean(raw)
df.to_csv('../data/telco_clean.csv', index=False)
print(df.shape, df.Churn.mean().round(4), df.TotalCharges.dtype)
df.describe().round(2)

(7043, 21) 0.2654 float64


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn
count,7043.00,7043.00,7043.00,7043.00,7043.00
mean,0.16,32.37,64.76,2279.73,0.27
std,0.37,24.56,30.09,2266.79,0.44
min,0.00,0.00,18.25,0.00,0.00
25%,0.00,9.00,35.50,398.55,0.00
50%,0.00,29.00,70.35,1394.55,0.00
75%,0.00,55.00,89.85,3786.60,1.00
max,1.00,72.00,118.75,8684.80,1.00


**Output:** `data/telco_clean.csv`, 7,043 rows, correct dtypes, binary target. Next: Phase 2, loading into SQLite as three tables (customers, services, billing).